# Elephino integer-width analysis

The hypothesis

$$
\max(\mathrm{Elephino}) = 2\,\texttt{leavesTotal} + 1
$$

is false. It is already too small for a $3\times3$ map, and a $2\times22$ map reaches 256 near the start of the canonical traversal even though the hypothesis predicts only 89.

What *is* proved here is the shape-independent safety bound

$$
\boxed{\max(\mathrm{Elephino})\leq \frac{N(N-1)}2+1},
\qquad N=\texttt{leavesTotal}.
$$

This bound is conservative, not generally exact. The exact maximum depends on `mapShape`, dimension order, and which transformed flow is executed, so `leavesTotal` alone cannot determine the smallest safe width once the conservative bound crosses a dtype boundary.

## What `Elephino` means in this project

Every current runtime use reduces to four quantities from Lunnon's gap stack:

| Project identifier | Lunnon identifier | Meaning |
|---|---|---|
| `gap1ndex` | `g` | Excluded end of the filtered, actual-gap stack |
| `gap1ndexCeiling` | `gg` | Excluded end of the possible-gap scan |
| `indexMiniGap` | `j` | Cursor scanning `[gap1ndex, gap1ndexCeiling)` |
| `gapRangeStart` | `gapter` | Saved stack position for each leaf |

The aliases are declared in [`theTypes.py`](../../theTypes.py), attached to state in [`dataBaskets.py`](../../dataBaskets.py), and manipulated by the canonical flow in [`daoOfMapFolding.py`](../../algorithms/daoOfMapFolding.py).

`gapsWhere` is deliberately *not* Elephino-valued: its elements are leaf labels, while its indices are Elephino values. The A007822 flow briefly reuses `indexMiniGap` as a leaf-linked-list cursor, but that extra role is bounded by $N$ and cannot dominate the gap cursor.

The TODO in [`makeJobTheorem2Numba.py`](../../someAssemblyRequired/makeJobTheorem2Numba.py) asks only about `gapRangeStart.max()`. That cannot establish a safe width for the shared scalar dtype: `gap1ndexCeiling` and `indexMiniGap` are often materially larger.

## The `+1` is not an indexing illusion

At active leaf $l$, the candidate gap labels are $0,\ldots,l-1$. Zero-indexing changes their labels, but not their cardinality: there are $l$ candidates.

`gap1ndexCeiling` is an excluded upper bound, and `indexMiniGap` finishes at that bound. Therefore a one-past value really is stored. The relevant `+1` comes from stack and interval semantics, not from mixing zero-indexed and one-indexed leaf names.

## Reproduce the counterexamples

The monitor below observes the unmodified canonical implementation. It uses the project's default arbitrary-precision Python integers so that the mathematical values remain visible. For a large case it raises a private sentinel exception at a requested threshold, avoiding the full enumeration.

In [1]:
from __future__ import annotations

from pprint import pprint
from sys import settrace
from types import FrameType
from typing import Any

from mapFolding.algorithms.daoOfMapFolding import doTheNeedful
from mapFolding.dataBaskets import MapFoldingState


class ThresholdElephinoReached(RuntimeError):
	pass


class ElephinoMonitor:
	def __init__(self, thresholdElephino: int | None = None) -> None:
		self.thresholdElephino = thresholdElephino
		self.maximumElephino: dict[str, int] = {
			'gap1ndex': 0,
			'gap1ndexCeiling': 0,
			'gapRangeStart': 0,
			'indexMiniGap': 0,
		}
		self.contextThreshold: dict[str, int] | None = None

	def monitorObservesState(self, state: MapFoldingState) -> None:
		self.maximumElephino['gap1ndex'] = max(self.maximumElephino['gap1ndex'], int(state.gap1ndex))
		self.maximumElephino['gap1ndexCeiling'] = max(self.maximumElephino['gap1ndexCeiling'], int(state.gap1ndexCeiling))
		self.maximumElephino['gapRangeStart'] = max(self.maximumElephino['gapRangeStart'], int(state.gapRangeStart.max()))
		self.maximumElephino['indexMiniGap'] = max(self.maximumElephino['indexMiniGap'], int(state.indexMiniGap))
		if self.thresholdElephino is not None and max(self.maximumElephino.values()) >= self.thresholdElephino:
			self.contextThreshold = {
				'groupsOfFolds': int(state.groupsOfFolds),
				'leaf1ndex': int(state.leaf1ndex),
				'gap1ndex': int(state.gap1ndex),
				'gap1ndexCeiling': int(state.gap1ndexCeiling),
				'gapRangeStartMaximum': int(state.gapRangeStart.max()),
				'indexMiniGap': int(state.indexMiniGap),
			}
			raise ThresholdElephinoReached

	def __call__(self, frame: FrameType, event: str, _argumentIgnored: Any) -> Any:
		if event in {'line', 'return'} and frame.f_code.co_filename.endswith('daoOfMapFolding.py'):
			state = frame.f_locals.get('state')
			if isinstance(state, MapFoldingState):
				self.monitorObservesState(state)
		return self


def instrumentationAnalyzesMapShape(
	mapShape: tuple[int, ...],
	thresholdElephino: int | None = None,
) -> tuple[MapFoldingState, ElephinoMonitor]:
	mapFoldingStateAnalyzed = MapFoldingState(mapShape)
	elephinoMonitor = ElephinoMonitor(thresholdElephino)
	try:
		settrace(elephinoMonitor)
		mapFoldingStateAnalyzed = doTheNeedful(mapFoldingStateAnalyzed)
	except ThresholdElephinoReached:
		pass
	finally:
		settrace(None)
	return mapFoldingStateAnalyzed, elephinoMonitor

In [2]:
mapFoldingState3x3, elephinoMonitor3x3 = instrumentationAnalyzesMapShape((3, 3))
pprint({
	'foldsTotal': int(mapFoldingState3x3.foldsTotal),
	'leavesTotal': int(mapFoldingState3x3.leavesTotal),
	'maximumElephino': elephinoMonitor3x3.maximumElephino,
	'hypothesizedMaximum': 2 * int(mapFoldingState3x3.leavesTotal) + 1,
}, sort_dicts=False)

{'foldsTotal': 1368,
 'leavesTotal': 9,
 'maximumElephino': {'gap1ndex': 15,
                     'gap1ndexCeiling': 21,
                     'gapRangeStart': 14,
                     'indexMiniGap': 21},
 'hypothesizedMaximum': 19}


For $(3,3)$, the canonical complete run reaches 21 in both the ceiling and cursor, while $2N+1=19$. This is the smallest counterexample found among map shapes whose dimensions are all at least two.

In [3]:
mapFoldingState2x22, elephinoMonitor2x22 = instrumentationAnalyzesMapShape((2, 22), thresholdElephino=256)
pprint({
	'leavesTotal': int(mapFoldingState2x22.leavesTotal),
	'firstContextAtOrAbove256': elephinoMonitor2x22.contextThreshold,
	'maximumObservedBeforeStopping': elephinoMonitor2x22.maximumElephino,
	'hypothesizedMaximum': 2 * int(mapFoldingState2x22.leavesTotal) + 1,
}, sort_dicts=False)

{'leavesTotal': 44,
 'firstContextAtOrAbove256': {'groupsOfFolds': 0,
                              'leaf1ndex': 44,
                              'gap1ndex': 251,
                              'gap1ndexCeiling': 256,
                              'gapRangeStartMaximum': 251,
                              'indexMiniGap': 252},
 'maximumObservedBeforeStopping': {'gap1ndex': 252,
                                   'gap1ndexCeiling': 256,
                                   'gapRangeStart': 251,
                                   'indexMiniGap': 254},
 'hypothesizedMaximum': 89}


The $(2,22)$ monitor stops before counting even one folding. At active leaf 44, `gapRangeStart` has already reached 251 and `gap1ndexCeiling` reaches 256. At that instant, `gapsWhere[255]` is a valid array access (the array has 1,937 elements), and its stored leaf label also fits. The first representability violation is solely the scalar ceiling increment from 255 to 256. A `uint8` ceiling cannot represent that value; under rollover semantics it becomes 0, after which the next candidate overwrites the wrong gap slot. The hypothesis predicts only 89.

A one-dimensional 32-leaf run also reaches 256 on an early path, while the hypothesis predicts 65.

## Additional exhaustive observations

These are complete canonical traversals, not samples:

| `mapShape` | $N$ | max `gapRangeStart` | max `gap1ndex` | max ceiling/cursor | $2N+1$ |
|---|---:|---:|---:|---:|---:|
| `(3, 3)` | 9 | 14 | 15 | 21 | 19 |
| `(2, 5)` | 10 | 14 | 15 | 22 | 21 |
| `(5, 2)` | 10 | 14 | 15 | 20 | 21 |
| `(12,)` | 12 | 35 | 36 | 36 | 25 |
| `(2, 6)` | 12 | 19 | 20 | 26 | 25 |
| `(3, 4)` | 12 | 21 | 22 | 29 | 25 |
| `(4, 3)` | 12 | 22 | 23 | 31 | 25 |
| `(2, 2, 3)` | 12 | 14 | 15 | 25 | 25 |
| `(2, 8)` | 16 | 34 | 35 | 43 | 33 |
| `(4, 4)` | 16 | 32 | 33 | 41 | 33 |

Two consequences matter:

1. The exact maximum is not a function of $N$ alone.
2. Dimension order can change the internal maximum even when it does not change `foldsTotal`, as `(2, 5)` versus `(5, 2)` demonstrates.

For one-dimensional shapes, exhaustive runs through $N=14$ matched $\lceil N^2/4\rceil$. That expression is not a universal bound: `(2, 3)` reaches 11 with $N=6$, exceeding $\lceil N^2/4\rceil=9$. For $(2,m)$, complete runs through $m=10$ also showed quadratic growth. These are useful empirical patterns, but they are not used as proofs or safety guarantees.

## Proof of a universal safe bound

Let $s_l=\texttt{gapRangeStart}[l-1]$ when leaf $l$ becomes active.

1. `gap1ndexCeiling` starts at $s_l$.
2. It advances only for the first occurrence of a candidate gap label.
3. Candidate labels lie in $\{0,\ldots,l-1\}$, so there are at most $l$ distinct candidates.
4. Hence `gap1ndexCeiling` $\leq s_l+l$.
5. Filtering retains a subset, so `gap1ndex` is no larger.
6. Before descending, the algorithm decrements `gap1ndex` and saves it as `gapRangeStart[l]`. Therefore

$$
\texttt{gapRangeStart}[l]
\leq \texttt{gapRangeStart}[l-1]+l-1.
$$

Since `gapRangeStart[0] = 0`, induction gives

$$
\texttt{gapRangeStart}[l]\leq\sum_{r=1}^{l}(r-1)=\frac{l(l-1)}2.
$$

At active leaf $l$,

$$
\max(\texttt{gap1ndex},\texttt{gap1ndexCeiling},\texttt{indexMiniGap})
\leq \frac{(l-1)(l-2)}2+l
=\frac{l(l-1)}2+1.
$$

This expression increases with $l$. Thus, for $l\leq N$,

$$
\boxed{B(N)=\frac{N(N-1)}2+1}
$$

bounds every current Elephino quantity. It is much tighter than Lunnon's $N^2+1$-element gap-array allocation, but it is still deliberately conservative.

## Convert the bound to a dtype width

For a nonnegative maximum $B$, the mathematical number of unsigned bits is

$$
\left\lceil\log_2(B+1)\right\rceil.
$$

Substituting the proved bound gives

$$
\boxed{
b(N)=\left\lceil
\log_2\left(\frac{N(N-1)}2+2\right)
\right\rceil
}.
$$

In Python, this is simply `B.bit_length()`.

In [4]:
def mathematicsComputesElephinoUpperBound(leavesTotal: int) -> int:
	if leavesTotal < 1:
		raise ValueError(f'`leavesTotal` must be positive: received {leavesTotal}.')
	return leavesTotal * (leavesTotal - 1) // 2 + 1


def computationSelectsStandardUnsignedWidth(leavesTotal: int) -> int:
	requiredBits = mathematicsComputesElephinoUpperBound(leavesTotal).bit_length()
	if requiredBits <= 8:
		return 8
	if requiredBits <= 16:
		return 16
	if requiredBits <= 32:
		return 32
	if requiredBits <= 64:
		return 64
	raise OverflowError(f'No standard NumPy unsigned integer can hold the proved Elephino bound requiring {requiredBits} bits.')


def computationSummarizesUnsignedWidth(leavesTotal: int) -> dict[str, int]:
	maximumElephinoSafe = mathematicsComputesElephinoUpperBound(leavesTotal)
	return {
		'leavesTotal': leavesTotal,
		'maximumElephinoSafe': maximumElephinoSafe,
		'requiredUnsignedBits': maximumElephinoSafe.bit_length(),
		'standardUnsignedWidth': computationSelectsStandardUnsignedWidth(leavesTotal),
	}


pprint(tuple(map(computationSummarizesUnsignedWidth, (9, 23, 24, 32, 44, 255))), sort_dicts=False)

({'leavesTotal': 9,
  'maximumElephinoSafe': 37,
  'requiredUnsignedBits': 6,
  'standardUnsignedWidth': 8},
 {'leavesTotal': 23,
  'maximumElephinoSafe': 254,
  'requiredUnsignedBits': 8,
  'standardUnsignedWidth': 8},
 {'leavesTotal': 24,
  'maximumElephinoSafe': 277,
  'requiredUnsignedBits': 9,
  'standardUnsignedWidth': 16},
 {'leavesTotal': 32,
  'maximumElephinoSafe': 497,
  'requiredUnsignedBits': 9,
  'standardUnsignedWidth': 16},
 {'leavesTotal': 44,
  'maximumElephinoSafe': 947,
  'requiredUnsignedBits': 10,
  'standardUnsignedWidth': 16},
 {'leavesTotal': 255,
  'maximumElephinoSafe': 32386,
  'requiredUnsignedBits': 15,
  'standardUnsignedWidth': 16})


### Immediate dtype consequences

- `uint8` is **universally certified** through $N=23$: $B(23)=254$.
- The universal certificate switches to `uint16` at $N=24$: $B(24)=277$.
- This cutoff is sufficient, not necessary. Some particular shapes with $N>23$ remain safe in `uint8`; `leavesTotal` alone cannot prove which ones.
- If `leavesTotal` itself fits in `uint8`, then $N\leq255$ and $B(N)\leq32386$. Therefore **every current Elephino value is universally safe in `uint16` and also in `int16`**.
- A signed `int8` universal certificate ends at $N=16$, because its positive maximum is 127.

For production dtype selection, the safe default is therefore:

```python
maximumElephinoSafe = leavesTotal * (leavesTotal - 1) // 2 + 1
requiredUnsignedBits = maximumElephinoSafe.bit_length()
```

Then select the first supported unsigned dtype whose width is at least `requiredUnsignedBits`.

## Why this does not completely solve the performance decision

The proved bound answers safety, but it does not always identify the smallest safe dtype for one particular shape. Examples with the same $N$ have different maxima, and dimension order matters. A tighter certificate must therefore use more information than `leavesTotal`—at least `mapShape` and the selected flow—or prove an additional structural theorem.

There is also a useful type-design split:

- `gapRangeStart` stores the post-decrement stack position.
- `gap1ndex` stores the filtered end.
- `gap1ndexCeiling` and `indexMiniGap` store the larger possible-gap excluded bound.

The current scalar alias combines the last three. If memory traffic or scalar width is critical, separating the saved stack dtype from the ceiling/cursor dtype may preserve a narrower array even when the scan scalars need another bit. That is an implementation option, not part of the mathematical proof.

## Literature and independent implementation check

Lunnon's 1971 paper defines the algorithm and describes `g`, `gg`, `j`, and `gapter` with the same roles used above. The declarations and gap-stack description are on journal page 79; the initialization and filtering loop are on page 80:

- W. F. Lunnon, [*Multi-dimensional map-folding*](https://doi.org/10.1093/comjnl/14.1.75), *The Computer Journal* 14(1), 75–80 (1971).
- The project's transcription: [`foldings.AA`](foldings.AA).
- The project's close Python translation: [`lunnonWhile.py`](lunnonWhile.py).

Lunnon allocates `gap[0:n*n]`; that is a conservative storage allocation, not a claimed tight value bound.

Sean Irvine's independent Java port, based on Fred Lunnon's C implementation, likewise allocates `new int[n * n + 1]` and repeats that `gg` and `g` are possible- and actual-gap counts offset by `gapter[l-1]`:

- [Pinned `A001415.java` source](https://github.com/archmageirvine/joeis/blob/317a90e4c4ad24cfb545a6520d70542f54916a15/src/irvine/oeis/a001/A001415.java)

The literature and independent implementation corroborate the semantics. Neither provides a tight fixed-width formula; the conservative bound in this notebook is derived directly from the algorithm's recurrence.

## Bottom line

1. `max(Elephino) == 2 * leavesTotal + 1` is false, independently of indexing convention.
2. `uint8` can silently fail in an ordinary multidimensional case: $(2,22)$ reaches 256 before the first completed folding.
3. The proved project-wide safety bound is $N(N-1)/2+1$.
4. The unsigned mathematical width is `((N * (N - 1)) // 2 + 1).bit_length()`.
5. When $N\leq255$, 16 bits always suffice; 8 bits are universally guaranteed only through $N=23$.
6. Beyond that universal cutoff, an exact smallest-width decision needs a shape/flow-specific theorem or analysis.